In [2]:
# -*- coding: utf-8 -*-
"""claude3.jpnyb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1zuE0vIccVjFsStIgHFyAbhgAS9x883Hd
"""

from google.colab import drive
drive.mount('/content/drive')

import os
import urllib.request
import zipfile

URL = "https://archive.ics.uci.edu/static/public/296/diabetes+130-us+hospitals+for+years+1999-2008.zip"
ZIP_PATH = "diabetes_130.zip"
EXTRACT_DIR = "diabetes_data"


def find_csv():
    """Return the path to diabetic_data.csv anywhere under the current folder, or None."""
    for root, _, files in os.walk("."):
        if "diabetic_data.csv" in files:
            return os.path.join(root, "diabetic_data.csv")
    return None


if find_csv() is None:
    try:
        print("Downloading dataset from UCI...")
        urllib.request.urlretrieve(URL, ZIP_PATH)
        with zipfile.ZipFile(ZIP_PATH) as z:
            z.extractall(EXTRACT_DIR)
    except Exception as e:
        print(f"Automatic download failed: {e}")
        print("Please upload the zip file you have.")
        from google.colab import files
        uploaded = files.upload()
        zip_name = next(iter(uploaded))
        with zipfile.ZipFile(zip_name) as z:
            z.extractall(EXTRACT_DIR)

csv_path = find_csv()
if csv_path is None:
    raise SystemExit("diabetic_data.csv was not found after download or upload.")
print("Dataset ready:", csv_path)

# -*- coding: utf-8 -*-
""" 30-day hospital readmission risk - Diabetes 130-US Hospitals (1999-2008) Corrected pipeline and clinical app. Design decisions (put these in the write-up): - Target: readmitted within 30 days ("<30") vs not ("NO" or ">30"). - Patients who died or went to hospice (discharge ids 11, 19, 20, 21) are removed, because they cannot be readmitted. - Split by patient_nbr: a patient's encounters stay in ONE split. Otherwise the model sees the same patient in training and testing, which inflates results. - Train 60% / validation 20% / test 20%. Threshold chosen on validation only. The test set is used once. - weight is dropped (97% missing). race and gender are excluded from the model and used only for the fairness audit. - Diagnosis codes are grouped into ICD-9 chapters (250.x = "Diabetes", etc.). Environment variables for testing: QUICK=1 small training subsample and fewer search iterations NO_LAUNCH=1 run the modelling and skip the Gradio app """

import glob
import os

import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier, VotingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, confusion_matrix, roc_auc_score, roc_curve
from sklearn.model_selection import GroupShuffleSplit, RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# ============================ CONFIG ============================
SEED = 42
QUICK = os.environ.get("QUICK") == "1"
NO_LAUNCH = os.environ.get("NO_LAUNCH") == "1"
DATA_PATH = "diabetic_data.csv"
DISCLAIMER = "This project is a prototype and is not medical advice."
TARGET_RAW = "readmitted"
EXPIRED_IDS = [11, 19, 20, 21]
DROP_COLS = ["encounter_id", "patient_nbr", "weight", TARGET_RAW]
EXCLUDE_FROM_MODEL = ["race", "gender"]          # used only for the fairness audit
TARGET_RECALL = None                             # None = Youden's J; or e.g. 0.60
THRESH_MIN, THRESH_MAX = 0.02, 0.50              # rare-event range (positives ~11%)
AGE_BINS = [0, 50, 70, 200]
AGE_LABELS = ["<50", "50-69", "70+"]
SAVE_PATH = "/content/drive/MyDrive/readmission_diabetes_cache.joblib"

ID_AS_CAT = ["admission_type_id", "discharge_disposition_id", "admission_source_id"]
DIAG_COLS = ["diag_1", "diag_2", "diag_3"]
DRUG_COLS = ["metformin", "repaglinide", "nateglinide", "chlorpropamide", "glimepiride",
             "acetohexamide", "glipizide", "glyburide", "tolbutamide", "pioglitazone",
             "rosiglitazone", "acarbose", "miglitol", "troglitazone", "tolazamide", "examide",
             "citoglipton", "insulin", "glyburide-metformin", "glipizide-metformin",
             "glimepiride-pioglitazone", "metformin-rosiglitazone", "metformin-pioglitazone"]
COMPUTED_NAMES = ["prior_visits", "num_drugs_active"]

TR_LABELS = {
    "age": "Patient Age (years)",
    "time_in_hospital": "Length of Stay (days)",
    "num_lab_procedures": "Lab Procedures This Visit",
    "num_procedures": "Non-lab Procedures This Visit",
    "num_medications": "Distinct Medications Prescribed",
    "number_outpatient": "Outpatient Visits in Prior Year",
    "number_emergency": "Emergency Visits in Prior Year",
    "number_inpatient": "Inpatient Admissions in Prior Year",
    "number_diagnoses": "Number of Diagnoses",
    "diag_1": "Primary Diagnosis Group",
    "diag_2": "Secondary Diagnosis Group",
    "diag_3": "Additional Diagnosis Group",
    "admission_type_id": "Admission Type",
    "discharge_disposition_id": "Discharge Disposition",
    "admission_source_id": "Admission Source",
    "payer_code": "Payer",
    "medical_specialty": "Admitting Medical Specialty",
    "max_glu_serum": "Max Glucose Serum Test Result",
    "A1Cresult": "HbA1c Test Result",
    "change": "Diabetes Medication Changed",
    "diabetesMed": "Diabetes Medication Prescribed",
    "prior_visits": "Total Prior Visits (computed)",
    "num_drugs_active": "Diabetes Drugs Active (computed)",
}
PRIMARY_COLS = ["age", "time_in_hospital", "number_inpatient", "number_emergency",
                "number_outpatient", "num_medications", "diag_1", "A1Cresult"]
# ================================================================


# ---------- 1. Cleaning helpers ----------
def age_to_years(s):
    """'[70-80)' -> 75.0. Numeric input passes through unchanged."""
    if isinstance(s, (int, float, np.integer, np.floating)):
        return float(s)
    lo, hi = str(s).strip("[)").split("-")
    return (float(lo) + float(hi)) / 2


def icd_group(code):
    """Group an ICD-9 diagnosis code into its standard chapter."""
    if pd.isna(code):
        return "Missing"
    s = str(code).strip()
    if s.upper().startswith(("V", "E")):
        return "External/Supplementary"
    try:
        v = float(s)
    except ValueError:
        return "Other"
    if 250 <= v < 251:
        return "Diabetes"
    if 390 <= v < 460 or v == 785:
        return "Circulatory"
    if 460 <= v < 520 or v == 786:
        return "Respiratory"
    if 520 <= v < 580 or v == 787:
        return "Digestive"
    if 580 <= v < 630 or v == 788:
        return "Genitourinary"
    if 140 <= v < 240:
        return "Neoplasms"
    if 710 <= v < 740:
        return "Musculoskeletal"
    if 800 <= v < 1000:
        return "Injury"
    return "Other"


def prep_raw(df):
    """Training-only cleaning: converts raw codes into model-ready values."""
    df = df.copy()
    df["age"] = df["age"].map(age_to_years)
    for c in DIAG_COLS:
        df[c] = df[c].map(icd_group)
    for c in ID_AS_CAT:
        df[c] = df[c].astype("Int64").astype(str).replace("<NA>", "Missing")
    return prep_common(df)


def prep_common(df):
    """Applied to training data AND app input: fill and type every field the same way."""
    df = df.copy()
    for c in df.columns:
        if c in NUMERIC_IN_APP:
            df[c] = pd.to_numeric(df[c], errors="coerce")
        else:
            df[c] = df[c].where(df[c].notna(), "Missing").astype(str)
    return df


NUMERIC_IN_APP = ["age", "time_in_hospital", "num_lab_procedures", "num_procedures",
                  "num_medications", "number_outpatient", "number_emergency",
                  "number_inpatient", "number_diagnoses"]


# ---------- 2. Feature engineering: ONE function for training and the app ----------
def add_features(df):
    df = df.copy()
    df["prior_visits"] = df[["number_outpatient", "number_emergency", "number_inpatient"]].sum(axis=1)
    drugs = [c for c in DRUG_COLS if c in df.columns]
    df["num_drugs_active"] = (df[drugs] != "No").sum(axis=1)
    return df


# ---------- 3. Load and prepare ----------
csv_path = DATA_PATH
if not os.path.exists(csv_path):
    matches = glob.glob("**/diabetic_data.csv", recursive=True)
    if not matches:
        raise SystemExit("diabetic_data.csv not found. Upload it or unzip the dataset first.")
    csv_path = matches[0]

raw = pd.read_csv(csv_path, na_values=["?"], low_memory=False)
n_before = len(raw)
raw = raw[~raw["discharge_disposition_id"].isin(EXPIRED_IDS)].reset_index(drop=True)
print(f"Rows: {n_before} -> {len(raw)} after removing expired/hospice discharges")
print(f"Unique patients: {raw['patient_nbr'].nunique()}")

groups = raw["patient_nbr"].values
y_all = (raw[TARGET_RAW] == "<30").astype(int).reset_index(drop=True)
print(f"Positive rate (readmitted <30 days): {y_all.mean():.3f}")

feat = add_features(prep_raw(raw))
X_all = feat.drop(columns=[c for c in DROP_COLS if c in feat.columns]).reset_index(drop=True)

MODEL_FEATURES = [c for c in X_all.columns if c not in EXCLUDE_FROM_MODEL]
NUM_COLS = X_all[MODEL_FEATURES].select_dtypes(include=[np.number]).columns.tolist()
CAT_COLS = [c for c in MODEL_FEATURES if c not in NUM_COLS]
X_all = X_all[MODEL_FEATURES]

computed = [c for c in COMPUTED_NAMES if c in MODEL_FEATURES]
UI_COLS = [c for c in MODEL_FEATURES if c not in computed]
print(f"Model features: {len(MODEL_FEATURES)} ({len(NUM_COLS)} numeric, {len(CAT_COLS)} categorical)")
print(f"Computed in app: {computed}")

# ---------- 4. Split by PATIENT, not by row ----------
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
tmp_idx, test_idx = next(gss.split(X_all, y_all, groups))
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)
tr_rel, val_rel = next(gss2.split(X_all.iloc[tmp_idx], y_all.iloc[tmp_idx], groups[tmp_idx]))
train_idx, val_idx = tmp_idx[tr_rel], tmp_idx[val_rel]

p_train, p_val, p_test = set(groups[train_idx]), set(groups[val_idx]), set(groups[test_idx])
assert not (p_train & p_val) and not (p_train & p_test) and not (p_val & p_test), "patient leakage"

if QUICK:   # small subsample for testing only
    train_idx = np.random.RandomState(SEED).choice(train_idx, 8000, replace=False)

X_train, y_train = X_all.iloc[train_idx], y_all.iloc[train_idx]
X_val, y_val = X_all.iloc[val_idx], y_all.iloc[val_idx]
X_test, y_test = X_all.iloc[test_idx].reset_index(drop=True), y_all.iloc[test_idx].reset_index(drop=True)
test_raw = raw.iloc[test_idx].reset_index(drop=True)
print(f"Patients -> train {len(p_train)} | val {len(p_val)} | test {len(p_test)}")
print(f"Rows -> train {len(train_idx)} | val {len(val_idx)} | test {len(test_idx)}")
print(f"Positive rate -> train {y_train.mean():.3f} | val {y_val.mean():.3f} | test {y_test.mean():.3f}")

# ---------- 5. Pipeline ----------
preprocessor = ColumnTransformer([
    ("num", SimpleImputer(strategy="median", add_indicator=True), NUM_COLS),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT_COLS),
])
n_est = 50 if QUICK else 300
ensemble = VotingClassifier(
    estimators=[
        ("rf", RandomForestClassifier(n_estimators=n_est, max_depth=12, class_weight="balanced",
                                      random_state=SEED, n_jobs=-1)),
        ("hgb", HistGradientBoostingClassifier(max_depth=6, learning_rate=0.03, max_iter=300,
                                               random_state=SEED)),
    ],
    voting="soft",
)
pipe = Pipeline([("prep", preprocessor), ("model", ensemble)])

# ---------- 6-8. Train ONCE, then reuse the saved model ----------
param_grid = {
    "model__rf__n_estimators": [100, 150, 200],
    "model__rf__max_depth": [6, 8, 10, 12],
    "model__rf__min_samples_split": [2, 5, 10],
    "model__hgb__max_depth": [3, 5, 7],
    "model__hgb__learning_rate": [0.01, 0.03, 0.05, 0.1],
    "model__hgb__max_iter": [100, 150, 200],
}
RETRAIN = os.environ.get("RETRAIN") == "1"       # set RETRAIN=1 only to force a new training run
naive = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
REQUIRED_KEYS = ["model", "threshold", "p_untuned_prob", "best_params", "cv_auc",
                 "model_features", "n_train", "seed"]


def cache_is_valid(c):
    return (all(k in c for k in REQUIRED_KEYS)
            and c["model_features"] == MODEL_FEATURES
            and c["n_train"] == len(X_train)
            and c["seed"] == SEED)


cache = joblib.load(SAVE_PATH) if (os.path.exists(SAVE_PATH) and not RETRAIN) else None
if cache is not None and cache_is_valid(cache):
    model = cache["model"]
    THRESHOLD = cache["threshold"]
    p_untuned_prob = cache["p_untuned_prob"]
    best_params = cache["best_params"]
    cv_auc = cache["cv_auc"]
    print("Loaded saved model from", SAVE_PATH, "- training skipped.")
else:
    if cache is not None:
        print("Saved model does not match this data or code. Retraining.")
    untuned = clone(pipe).fit(X_train, y_train)
    p_untuned_prob = untuned.predict_proba(X_test)[:, 1]

    search = RandomizedSearchCV(
        pipe, param_distributions=param_grid, n_iter=2 if QUICK else 15, scoring="roc_auc",
        cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED),
        random_state=SEED, n_jobs=-1, verbose=1,
    )
    search.fit(X_train, y_train)
    model = search.best_estimator_
    best_params, cv_auc = search.best_params_, search.best_score_

    p_val_prob = model.predict_proba(X_val)[:, 1]
    fpr, tpr, thr = roc_curve(y_val, p_val_prob)
    if TARGET_RECALL is None:
        THRESHOLD = float(thr[np.argmax(tpr - fpr)])
    else:
        ok = thr[tpr >= TARGET_RECALL]
        THRESHOLD = float(ok.max()) if len(ok) else float(thr[np.argmax(tpr - fpr)])
    THRESHOLD = round(float(np.clip(THRESHOLD, THRESH_MIN, THRESH_MAX)), 2)

    if not QUICK:   # never save a quick-test model over the real one
        joblib.dump({
            "model": model, "threshold": THRESHOLD, "p_untuned_prob": p_untuned_prob,
            "best_params": best_params, "cv_auc": cv_auc,
            "model_features": MODEL_FEATURES, "n_train": len(X_train), "seed": SEED,
        }, SAVE_PATH)
        print("Trained once and saved to", SAVE_PATH)

print("Best CV ROC-AUC (training):", round(cv_auc, 4))
print("Best parameters:", best_params)
print(f"Threshold chosen on validation: {THRESHOLD:.2f}")

# ---------- 9. Final evaluation on TEST set (run once) ----------
p_test_prob = model.predict_proba(X_test)[:, 1]
p_naive = naive.predict(X_test)


def summarize(name, y_true, pred, prob=None):
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    n = len(y_true)
    row = {
        "model": name, "n": n, "TP": tp, "FP": fp, "FN": fn, "TN": tn,
        "accuracy": round((tp + tn) / n, 4),
        "recall": round(tp / (tp + fn), 4) if (tp + fn) else 0.0,
        "specificity": round(tn / (tn + fp), 4) if (tn + fp) else 0.0,
        "precision": round(tp / (tp + fp), 4) if (tp + fp) else np.nan,
        "flag_rate": round((tp + fp) / n, 4),
    }
    if prob is not None:
        row["roc_auc"] = round(roc_auc_score(y_true, prob), 4)
        row["pr_auc"] = round(average_precision_score(y_true, prob), 4)
    return row


results = pd.DataFrame([
    summarize("Naive baseline (all LOW)", y_test, p_naive),
    summarize("Untuned ensemble @ 0.50", y_test, (p_untuned_prob >= 0.5).astype(int), p_untuned_prob),
    summarize(f"Tuned ensemble @ {THRESHOLD:.2f}", y_test, (p_test_prob >= THRESHOLD).astype(int), p_test_prob),
])
print(f"\nNote: a random classifier has PR-AUC about {y_test.mean():.3f} (= positive rate).")
print("=== TEST RESULTS (held-out, evaluated once) ===")
print(results.to_string(index=False))

# ---------- 10. Fairness audit on TEST set ----------
test_eval = X_test.copy()
test_eval["y_true"] = y_test.values
test_eval["y_prob"] = p_test_prob
test_eval["y_pred"] = (p_test_prob >= THRESHOLD).astype(int)
test_eval["race"] = test_raw["race"].where(test_raw["race"].notna(), "Missing").astype(str).values
test_eval["gender"] = test_raw["gender"].where(test_raw["gender"].notna(), "Unknown").astype(str).values
test_eval["age_band"] = pd.cut(test_eval["age"], AGE_BINS, labels=AGE_LABELS, right=False).astype(str)


def group_report(frame, col):
    rows = [summarize(f"{col}={g}", sub["y_true"].values, sub["y_pred"].values)
            for g, sub in frame.groupby(col)]
    rep = pd.DataFrame(rows)
    rep = rep[rep["n"] >= 30]
    print(f"\n--- Fairness by {col} (groups with n >= 30) ---")
    print(rep.to_string(index=False))
    if len(rep):
        print(f"Recall gap: {rep['recall'].max() - rep['recall'].min():.3f} | "
              f"Flag-rate gap: {rep['flag_rate'].max() - rep['flag_rate'].min():.3f}")
    return rep


for col in ["race", "gender", "age_band"]:
    group_report(test_eval, col)

# ---------- 11. Case presets from the TEST set (labels verified by the model) ----------
def to_ui_values(row):
    out = {}
    for c in UI_COLS:
        v = row[c]
        out[c] = float(v) if c in NUM_COLS else str(v)
    return out


fn_cases = test_eval[(test_eval.y_true == 1) & (test_eval.y_pred == 0)].sort_values("y_prob", ascending=False)
fp_cases = test_eval[(test_eval.y_true == 0) & (test_eval.y_pred == 1)].sort_values("y_prob", ascending=False)
PRESETS = {}
PRESET_TRUTH = {"FN": "readmitted within 30 days (model missed it)",
                "FP": "not readmitted within 30 days (false alarm)"}
if len(fn_cases):
    PRESETS["FN"] = to_ui_values(fn_cases.iloc[0])
if len(fp_cases):
    PRESETS["FP"] = to_ui_values(fp_cases.iloc[0])
DEFAULTS = PRESETS.get("FN", {})

# ---------- 12. App helpers ----------
def build_row(values):
    """Turns the app's inputs into a model row. Same functions as training."""
    ui = pd.DataFrame([dict(zip(UI_COLS, values))])
    ui = prep_common(ui)
    fe = add_features(ui)
    return fe.reindex(columns=MODEL_FEATURES)


def metrics_text(threshold):
    pred = (p_test_prob >= threshold).astype(int)
    r = summarize("", y_test.values, pred)
    return (
        f"Held-out TEST set ({r['n']} encounters) at decision threshold {threshold:.2f}\n"
        f"- FN (Missed Readmission <30d): {r['FN']} | - FP (False Alarms): {r['FP']}\n"
        f"- TP (Correctly Flagged): {r['TP']} | - TN (Correctly Flagged Low): {r['TN']}\n"
        f"- Accuracy: {r['accuracy']:.3f} | Recall: {r['recall']:.3f} | "
        f"Precision: {r['precision']:.3f} | Flagged: {r['flag_rate']:.1%}"
    )


def predict(threshold, *values):
    row = build_row(values)
    prob = float(model.predict_proba(row)[0, 1])
    label = ("HIGH 30-DAY READMISSION RISK WARNING (1)" if prob >= threshold
             else "LOW / MODERATE 30-DAY READMISSION RISK (0)")
    text = f"- {label} - Estimated probability: {prob * 100:.1f}% ({prob:.3f})"
    return text, metrics_text(threshold), DISCLAIMER


def run_preset(which, threshold):
    preset = PRESETS.get(which)
    if preset is None:
        msg = f"No {which} case exists in the test set at this threshold."
        return [gr.update() for _ in UI_COLS] + [msg, metrics_text(threshold), DISCLAIMER]
    vals = [preset[c] for c in UI_COLS]
    text, metrics, notice = predict(threshold, *vals)
    text += f"\n- Verified test-set outcome: {PRESET_TRUTH[which]}"
    return vals + [text, metrics, notice]


# ---------- 13. Export app assets (lets app.py run without retraining) ----------
if not QUICK:
    import json
    assets = {
        "ui_cols": UI_COLS,
        "num_cols": NUM_COLS,
        "model_features": MODEL_FEATURES,
        "choices": {c: sorted(X_all[c].astype(str).unique().tolist())
                    for c in UI_COLS if c not in NUM_COLS},
        "defaults": DEFAULTS,
        "presets": PRESETS,
        "preset_truth": PRESET_TRUTH,
        "threshold": THRESHOLD,
        "test_labels": [int(v) for v in y_test.values],
        "test_probs": [float(v) for v in p_test_prob],
    }
    with open("app_assets.json", "w") as f:
        json.dump(assets, f)
    print("Wrote app_assets.json")

# ---------- 14. Gradio app (skipped when NO_LAUNCH=1) ----------
def build_app():
    import gradio as gr

    def make_component(col):
        label = TR_LABELS.get(col, col.replace("_", " ").title())
        default = DEFAULTS.get(col)
        if col in NUM_COLS:
            return gr.Number(value=default, label=label)
        choices = sorted(X_all[col].astype(str).unique().tolist())
        return gr.Dropdown(choices=choices, value=default, label=label)

    with gr.Blocks(title="Diabetes Readmission Risk Portal") as demo:
        gr.Markdown("# 30-Day Readmission Risk Portal (Diabetes Patients)")
        gr.Markdown(
            "**Designed for discharge coordinators and nurses** to estimate the chance that a diabetic "
            "patient is readmitted within 30 days. Preset cases come from the held-out test set. "
            "Changing a field does not update the result until you press **Evaluate**."
        )
        gr.Markdown(f"> **Legal Disclaimer:** {DISCLAIMER}")

        with gr.Row():
            btn_fn = gr.Button("Load Case 1: Missed Readmission (False Negative)")
            btn_fp = gr.Button("Load Case 2: False Alarm (False Positive)")

        th_slider = gr.Slider(
            minimum=THRESH_MIN, maximum=THRESH_MAX, step=0.01, value=THRESHOLD,
            label="Decision Threshold (lowering reduces missed readmissions but increases alerts)",
        )

        gr.Markdown("### Primary Clinical Variables")
        primary = [c for c in PRIMARY_COLS if c in UI_COLS]
        secondary = [c for c in UI_COLS if c not in primary]
        comps = {}
        with gr.Row():
            for c in primary[:4]:
                comps[c] = make_component(c)
        with gr.Row():
            for c in primary[4:]:
                comps[c] = make_component(c)
        with gr.Accordion("Optional Demographics & Secondary Indicators", open=False):
            for c in secondary:
                comps[c] = make_component(c)
        ordered = [comps[c] for c in UI_COLS]

        run_btn = gr.Button("Evaluate Readmission Risk", variant="primary")
        out_risk = gr.Textbox(label="1. Calculated Risk Class", lines=3)
        out_metrics = gr.Textbox(value=metrics_text(THRESHOLD),
                                 label="2. Held-out Test-Set Metrics at the Current Threshold",
                                 lines=4, interactive=False)
        out_notice = gr.Textbox(value=DISCLAIMER, label="3. Legal Notice", interactive=False)
        outputs = [out_risk, out_metrics, out_notice]

        run_btn.click(predict, inputs=[th_slider] + ordered, outputs=outputs)
        th_slider.change(predict, inputs=[th_slider] + ordered, outputs=outputs)
        btn_fn.click(lambda t: run_preset("FN", t), inputs=[th_slider], outputs=ordered + outputs)
        btn_fp.click(lambda t: run_preset("FP", t), inputs=[th_slider], outputs=ordered + outputs)
    return demo


if not NO_LAUNCH:
    build_app().launch(share=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset ready: ./diabetes_data/diabetic_data.csv
Rows: 101766 -> 100114 after removing expired/hospice discharges
Unique patients: 70439
Positive rate (readmitted <30 days): 0.113
Model features: 46 (11 numeric, 35 categorical)
Computed in app: ['prior_visits', 'num_drugs_active']
Patients -> train 42263 | val 14088 | test 14088
Rows -> train 59964 | val 20075 | test 20075
Positive rate -> train 0.112 | val 0.116 | test 0.115
Loaded saved model from /content/drive/MyDrive/readmission_diabetes_cache.joblib - training skipped.
Best CV ROC-AUC (training): 0.6725
Best parameters: {'model__rf__n_estimators': 150, 'model__rf__min_samples_split': 5, 'model__rf__max_depth': 10, 'model__hgb__max_iter': 150, 'model__hgb__max_depth': 7, 'model__hgb__learning_rate': 0.03}
Threshold chosen on validation: 0.30

Note: a random classifier has PR-AUC about 0.115 (= positive r

In [4]:
import sklearn, pandas, numpy, gradio; print(sklearn.__version__, pandas.__version__, numpy.__version__, gradio.__version__)

1.6.1 2.2.3 2.4.6 6.31.0
